# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [11]:
!pip install -U langchain langchain_openai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00


In [12]:
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAI
from langchain import HuggingFaceHub
from langchain_core.runnables import Runnable
from langchain_core.prompts import PromptTemplate
from langchain.chains import SimpleSequentialChain, LLMChain, LLMMathChain, TransformChain, SequentialChain
from langchain_core.output_parsers import StrOutputParser
from langchain import FewShotPromptTemplate
from langchain.callbacks import get_openai_callback
import inspect
import re
import os

In [13]:
import warnings
warnings.filterwarnings('ignore')

In [25]:
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [16]:
# import os

# from dotenv import load_dotenv, find_dotenv
# _ = load_dotenv(find_dotenv())

# OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [17]:
!pip install pandas

In [19]:
import pandas as pd
df = pd.read_csv('/content/Data.csv')

In [ ]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [ ]:
!pip install langchain_community

In [20]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [26]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.5)


In [28]:
prompt = ChatPromptTemplate.from_template(
    "Write a short marketing description for the product: {product}"
)

In [29]:
chain = LLMChain(llm=llm, prompt=prompt)

In [30]:
product = "Smart Watch for Fitness Tracking"
chain.run(product)

'Stay on top of your fitness goals with our Smart Watch for Fitness Tracking. This sleek and stylish watch not only tells time, but also tracks your steps, calories burned, heart rate, and sleep patterns. With customizable fitness goals and notifications to keep you motivated, this smart watch is the perfect companion for your active lifestyle. Get yours today and take your fitness to the next level!'

## SimpleSequentialChain

In [31]:
from langchain.chains import SimpleSequentialChain

In [32]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Based on the product description '{product}', generate a detailed and engaging product summary."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [33]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Using the following product summary:\n\n{product_summary}\n\nWrite a creative social media advertisement targeting young professionals."
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [34]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [35]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Introducing the ultimate companion for your fitness journey - the Smart Watch for Fitness Tracking. This sleek and stylish smart watch is packed with features to help you reach your fitness goals and live a healthier lifestyle.

With advanced fitness tracking capabilities, this smart watch monitors your heart rate, steps taken, calories burned, and even tracks your sleep patterns. Stay motivated and on track with real-time updates on your progress throughout the day.

But that's not all - this smart watch also boasts smart notifications, allowing you to stay connected with calls, texts, and notifications right on your wrist. And with its long-lasting battery life, you can wear it all day and night without worrying about constantly recharging.

Whether you're a seasoned athlete or just starting your fitness journey, the Smart Watch for Fitness Tracking is the perfect tool to help you stay active, motivated, and on top of your health and we

"🌟 Calling all young professionals! 🌟 Take your fitness journey to the next level with our Smart Watch for Fitness Tracking. 🏃\u200d♂️🔥 Monitor your heart rate, track your steps, and stay on top of your health goals with this sleek and stylish accessory. 💪\n\nStay connected and motivated with smart notifications right on your wrist, while enjoying long-lasting battery life that keeps up with your active lifestyle. ⌚️ Don't wait any longer - get your hands on the ultimate fitness companion and crush your goals! 🏋️\u200d♀️✨ #FitnessGoals #SmartWatch #HealthyLiving #TechFitness #YoungProfessionals."

**Repeat the above twice for different products**

## SequentialChain

In [36]:
from langchain.chains import SequentialChain

In [38]:
llm = ChatOpenAI(temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
    "Translate the following product review to English:\n\n{review}"
)
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    output_key="translated_review"
)

In [39]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following product review in 1-2 sentences:\n\n{translated_review}"
)

chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
                    )


In [40]:
# prompt template 3: translate to english or other language

third_prompt = ChatPromptTemplate.from_template(
    "Translate the following product review to {language}:\n\n{review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="translated_review" )

In [41]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "Based on the following translated review:\n\n{translated_review}\n\n"
    "And its summary:\n\n{summary}\n\n"
    "Write a friendly follow-up message to thank the user and invite them to share more feedback."
)
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key="followup_message" )


In [42]:
# overall_chain: input= Review
# and output= English_Review,summary, followup_message


overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_four],
    input_variables=["review"],
    output_variables=["translated_review", "summary", "followup_message"],
    verbose=True )

In [43]:
review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'translated_review': "I find the taste mediocre. The foam doesn't hold, it's weird. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?",
 'summary': "The reviewer finds the taste of the product mediocre and notes that the foam doesn't hold well, leading them to question if they received an old batch or a counterfeit product.",
 'followup_message': 'Thank you for sharing your thoughts on the product. We appreciate your feedback and would love to hear more about your experience. Please feel free to reach out with any additional comments or concerns you may have. Your input is valuable to us as we strive to improve our products and provide the best possible experience for our customers. Thank you again for taking the time to share your thoughts.'}

**Repeat the above twice for different products or reviews**

## Router Chain

In [44]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [45]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [46]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [47]:
llm = ChatOpenAI(temperature=0)

In [48]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [49]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [50]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [51]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [52]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [53]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures."

In [54]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'2 + 2 equals 4.'

In [55]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells. \n\nAdditionally, DNA is constantly being used by cells to carry out essential processes such as protein synthesis, cell growth, and repair. Having DNA in every cell allows for the coordination of these processes and ensures the proper functioning of the organism as a whole.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**